In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))          # notebooks/ -> repo root
from config import directories

import pandas as pd
from sqlalchemy import create_engine

DB_PATH = directories.INTERIM_DATA / "panchayat_1.duckdb"
engine = create_engine(f"duckdb:///{DB_PATH}")

def q(sql):
    """Run SQL and return a DataFrame."""
    return pd.read_sql(sql, engine)

# confirm what's there
print(q("SHOW TABLES").to_string(index=False))

In [ ]:
for t in ["gram_panchayat", "plan", "planned_activity", "activity_expenditure",
          "voucher", "activity_voucher"]:
    print(f"{t:24} {q(f'SELECT count(*) FROM {t}').iloc[0,0]:>7}")

In [ ]:
# Q1: "How many activities did each gram panchayat plan, and what did they
#      budget, year by year?"
q("""
SELECT g.gp_name, a.fiscal_year,
       count(*) AS activities,
       sum(a.total_cost) AS planned_cost
FROM planned_activity a
JOIN gram_panchayat g USING (gp_lgd_code)
GROUP BY 1,2 ORDER BY 1,2
""")

In [ ]:
# Q2: "For Andhrua, how much was planned versus actually spent each year,
#      and what's the utilisation rate?"
q("""
-- Expenditure is one-to-many on activity_code. Joining first repeats
-- total_cost per expenditure row and makes count(*) count expenditure rows,
-- so planned cost and utilisation are both inflated. Aggregate to one row
-- per activity before joining.
SELECT a.fiscal_year,
       count(*) AS activities,
       sum(a.total_cost) AS planned,
       sum(e.total_expenditure) AS spent,
       round(100.0*sum(e.total_expenditure)/nullif(sum(a.total_cost),0),1) AS util_pct
FROM planned_activity a
LEFT JOIN (
    SELECT activity_code, sum(total_expenditure) AS total_expenditure
    FROM activity_expenditure
    GROUP BY activity_code
) e USING (activity_code)
WHERE a.gp_lgd_code='119598'
GROUP BY 1 ORDER BY 1
""")

In [ ]:
# Q1: "How many activities did Andhrua plan each year, and what did it budget?"
q("""
SELECT a.fiscal_year,
       count(*) AS activities,
       sum(a.total_cost) AS planned_cost
FROM planned_activity a
WHERE a.gp_lgd_code = '119598'
GROUP BY 1 ORDER BY 1
""")

In [ ]:
# Q2: "For Andhrua, how much was planned versus actually spent each year,
#      and what's the utilisation rate?"
q("""
-- Expenditure is one-to-many on activity_code. Joining first repeats
-- total_cost per expenditure row and makes count(*) count expenditure rows,
-- so planned cost and utilisation are both inflated. Aggregate to one row
-- per activity before joining.
SELECT a.fiscal_year,
       count(*) AS activities,
       sum(a.total_cost) AS planned,
       sum(e.total_expenditure) AS spent,
       round(100.0*sum(e.total_expenditure)/nullif(sum(a.total_cost),0),1) AS util_pct
FROM planned_activity a
LEFT JOIN (
    SELECT activity_code, sum(total_expenditure) AS total_expenditure
    FROM activity_expenditure
    GROUP BY activity_code
) e USING (activity_code)
WHERE a.gp_lgd_code = '119598'
GROUP BY 1 ORDER BY 1
""")

In [ ]:
# Q3: "What were Andhrua's total receipts and payments in each financial year?"
q("""
SELECT fiscal_year, direction, count(*) AS vouchers, sum(amount) AS total
FROM voucher
WHERE gp_lgd_code = '119598'
GROUP BY 1,2 ORDER BY 1,2
""")

In [ ]:
# Q4: "What are Andhrua's ten largest single payments?"
q("""
SELECT v.fiscal_year, v.voucher_no, v.type, v.date, v.amount
FROM voucher v
WHERE v.gp_lgd_code = '119598' AND v.direction = 'payment'
ORDER BY v.amount DESC
LIMIT 10
""")

In [ ]:
# Q6: "How much of Andhrua's funding is earmarked for General, SC, and ST
#      categories each year?"
q("""
SELECT a.fiscal_year,
       sum(f.fund_tied_general + f.fund_untied_general) AS general,
       sum(f.fund_tied_sc + f.fund_untied_sc)           AS sc,
       sum(f.fund_tied_st + f.fund_untied_st)           AS st,
       sum(f.fund_amount_total)                         AS total
FROM activity_fund f
JOIN planned_activity a USING (activity_code)
WHERE a.gp_lgd_code = '119598'
GROUP BY 1 ORDER BY 1
""")

In [ ]:
# Q7: "Show me Andhrua's 2025-26 activities traced all the way to the
#      individual vouchers that paid for them."
q("""
SELECT a.activity_code, a.activity_name, a.total_cost,
       e.total_expenditure, av.voucher_no, av.voucher_cost, v.date
FROM planned_activity a
JOIN activity_expenditure e USING (activity_code)
JOIN activity_voucher av USING (expenditure_id)
LEFT JOIN voucher v USING (voucher_pk)
WHERE a.gp_lgd_code = '119598' AND a.fiscal_year = '2025-2026'
ORDER BY av.voucher_cost DESC
LIMIT 15
""")

In [ ]:
#open updated database
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from config import directories

from sqlalchemy import create_engine

DB_PATH = directories.INTERIM_DATA / "panchayat_1.duckdb"
engine = create_engine(f"duckdb:///{DB_PATH}")
def q(sql): return pd.read_sql(sql, engine)

tables = q("SHOW TABLES")
print(f"{len(tables)} tables (expect 19)\n")
print(tables.to_string(index=False))

In [ ]:
#check tables are connected 
q("""
SELECT a.activity_code, a.activity_name,
       aa.adm_approval_no, aa.adm_approval_sanction_date,
       aa.work_proposed_cost, ta.tec_approval_cost,
       e.total_expenditure,
       count(pp.row_id) AS progress_records
FROM planned_activity a
JOIN admin_approval aa           USING (activity_code)
LEFT JOIN technical_approval ta  USING (activity_code)
LEFT JOIN activity_expenditure e USING (activity_code)
LEFT JOIN physical_progress pp   USING (activity_code)
WHERE a.gp_lgd_code = '119598'
GROUP BY 1,2,3,4,5,6,7
ORDER BY aa.adm_approval_sanction_date DESC
LIMIT 10
""")